# 02. SimPy 入门教程

这一章用小例子串起 SimPy 的基本用法。目标不是覆盖所有 API，而是让你能从零写出一个正确的离散事件仿真模型。

## 安装和导入

如果项目环境还没有安装 SimPy：

```bash
pip install simpy
```

代码中导入：

```python
import simpy
```

在本仓库中，如果已经包含本地 `simpy` 包，运行示例时 Python 会优先导入当前工程中的源码。阅读和调试本仓库 SimPy 源码时，这通常是期望行为。


## 第一个进程


In [1]:
import simpy


def car(env):
    print(f"{env.now}: 汽车开始停车")
    yield env.timeout(5)
    print(f"{env.now}: 汽车开始行驶")


env = simpy.Environment()
env.process(car(env))
env.run()

0: 汽车开始停车
5: 汽车开始行驶


解释：

- `car(env)` 是一个生成器函数，因为它内部有 `yield`。
- `env.process(car(env))` 把生成器包装成 SimPy 进程。
- `yield env.timeout(5)` 让进程暂停 5 个仿真时间单位。
- `env.run()` 运行仿真，直到没有事件可处理。

## 重复行为

很多仿真对象会不断循环，例如车辆停车、行驶、再停车：

In [2]:
import simpy


def car(env):
    while True:
        print(f"{env.now}: 停车")
        yield env.timeout(5)

        print(f"{env.now}: 行驶")
        yield env.timeout(2)


env = simpy.Environment()
env.process(car(env))
env.run(until=15)

0: 停车
5: 行驶
7: 停车
12: 行驶
14: 停车


`env.run(until=15)` 会运行到仿真时间 15。注意：当 `until` 是数字时，SimPy 在到达这个时间点后停止，不会继续处理这个时间点之后的事件。


## 多个进程并发

SimPy 进程是协作式并发。多个进程通过事件交替恢复：


In [3]:
import simpy


def car(env, name, parking_time, driving_time):
    while True:
        print(f"{env.now}: {name} 停车")
        yield env.timeout(parking_time)
        print(f"{env.now}: {name} 行驶")
        yield env.timeout(driving_time)


env = simpy.Environment()
env.process(car(env, "car-1", 5, 2))
env.process(car(env, "car-2", 3, 4))
env.run(until=12)

0: car-1 停车
0: car-2 停车
3: car-2 行驶
5: car-1 行驶
7: car-2 停车
7: car-1 停车
10: car-2 行驶


这里没有线程，也没有真实并行。SimPy 只是按事件时间顺序恢复不同进程。

## 进程等待另一个进程

进程本身也是事件，所以一个进程可以等待另一个进程结束。

In [4]:
import simpy


def charge(env, duration):
    print(f"{env.now}: 开始充电")
    yield env.timeout(duration)
    print(f"{env.now}: 充电完成")
    return "battery-full"


def car(env):
    result = yield env.process(charge(env, 5))
    print(f"{env.now}: 收到充电结果 {result}")


env = simpy.Environment()
env.process(car(env))
env.run()

0: 开始充电
5: 充电完成
5: 收到充电结果 battery-full


当 `charge()` 返回时，等待它的 `car()` 会收到返回值。


## 中断进程

进程可以被另一个进程中断。被中断的进程会在当前等待点收到 `simpy.Interrupt` 异常。


In [5]:
import simpy


def charge(env):
    try:
        print(f"{env.now}: 开始长时间充电")
        yield env.timeout(10)
        print(f"{env.now}: 充电完成")
    except simpy.Interrupt as interrupt:
        print(f"{env.now}: 充电被中断，原因: {interrupt.cause}")


def driver(env, charging_proc):
    yield env.timeout(3)
    charging_proc.interrupt("需要出发")


env = simpy.Environment()
charging = env.process(charge(env))
env.process(driver(env, charging))
env.run()

0: 开始长时间充电
3: 充电被中断，原因: 需要出发


中断适合建模：

- 任务取消。
- 更高优先级任务抢占。
- 设备故障。
- 用户离开队列。
- 调度器终止当前执行。

## 手动事件

`env.event()` 可以创建一个普通事件。这个事件不会自动触发，必须由代码调用 `succeed()` 或 `fail()`。


In [6]:
import simpy


def waiter(env, signal):
    print(f"{env.now}: 等待信号")
    value = yield signal
    print(f"{env.now}: 收到信号 {value}")


def sender(env, signal):
    yield env.timeout(4)
    signal.succeed("ready")


env = simpy.Environment()
signal = env.event()
env.process(waiter(env, signal))
env.process(sender(env, signal))
env.run()

0: 等待信号
4: 收到信号 ready


手动事件适合表达一次性通知。例如“初始化完成”“外部条件满足”“某个批次凑齐”。


## 条件事件：等待任意一个事件

使用 `|` 或 `simpy.events.AnyOf` 可以等待多个事件中的任意一个。


In [7]:
import simpy


def customer(env, service_done):
    patience = env.timeout(3)
    result = yield service_done | patience

    if service_done in result:
        print(f"{env.now}: 服务完成")
    else:
        print(f"{env.now}: 顾客等不及离开")


def service(env, service_done):
    yield env.timeout(5)
    service_done.succeed()


env = simpy.Environment()
done = env.event()
env.process(customer(env, done))
env.process(service(env, done))
env.run()

3: 顾客等不及离开


常见用途：

- 服务完成或超时，谁先发生听谁。
- 任务完成或取消，谁先发生听谁。
- 请求成功或截止时间到达。

## 条件事件：等待所有事件

使用 `&` 或 `simpy.events.AllOf` 可以等待所有事件都完成。


In [8]:
import simpy


def prepare(env):
    data = env.timeout(2, value="data-ready")
    model = env.timeout(5, value="model-ready")

    results = yield data & model
    print(f"{env.now}: 全部准备完成: {list(results.values())}")


env = simpy.Environment()
env.process(prepare(env))
env.run()

5: 全部准备完成: ['data-ready', 'model-ready']


适合建模：

- 多个前置任务全部完成后再启动。
- 批处理需要收齐多个分片。
- 资源和数据都可用后才执行。

## 使用 Resource 表达有限资源


In [9]:
import simpy


def customer(env, name, counter, service_time):
    print(f"{env.now}: {name} 到达")
    with counter.request() as req:
        yield req
        print(f"{env.now}: {name} 开始服务")
        yield env.timeout(service_time)
        print(f"{env.now}: {name} 离开")


env = simpy.Environment()
counter = simpy.Resource(env, capacity=1)

env.process(customer(env, "A", counter, 4))
env.process(customer(env, "B", counter, 2))
env.run()

0: A 到达
0: B 到达
0: A 开始服务
4: A 离开
4: B 开始服务
6: B 离开


`with counter.request() as req` 的好处是：离开 `with` 代码块时，资源会自动释放。


## 动态创建进程

到达过程通常由一个生成器不断创建顾客、请求或任务：

In [10]:
import random
import simpy


def customer(env, name, counter):
    arrive = env.now
    with counter.request() as req:
        yield req
        wait = env.now - arrive
        service_time = random.uniform(1, 3)
        print(f"{env.now:.1f}: {name} 等待 {wait:.1f}, 服务 {service_time:.1f}")
        yield env.timeout(service_time)


def source(env, counter):
    for i in range(5):
        env.process(customer(env, f"customer-{i}", counter))
        yield env.timeout(random.expovariate(1.0))


random.seed(7)
env = simpy.Environment()
counter = simpy.Resource(env, capacity=2)
env.process(source(env, counter))
env.run()

0.0: customer-0 等待 0.0, 服务 1.3
0.4: customer-1 等待 0.0, 服务 1.1
1.4: customer-2 等待 0.0, 服务 1.7
2.2: customer-3 等待 0.0, 服务 2.0
3.2: customer-4 等待 0.9, 服务 1.9


这个模式非常常见：

- `source()` 负责产生请求。
- `customer()` 负责描述单个请求生命周期。
- `counter` 负责限制并发服务能力。

## 记录统计指标

SimPy 不会自动保存业务统计。通常需要自己维护列表或日志。


In [13]:
import simpy


def customer(env, name, counter, waits):
    arrive = env.now
    with counter.request() as req:
        yield req
        wait = env.now - arrive
        waits.append(wait)
        print(f"{name} 等待时间: {wait}")
        yield env.timeout(2)


env = simpy.Environment()
counter = simpy.Resource(env, capacity=1)
waits = []

for i in range(3):
    env.process(customer(env, f"C{i}", counter, waits))

env.run()
print("平均等待时间:", sum(waits) / len(waits))

C0 等待时间: 0
C1 等待时间: 2
C2 等待时间: 4
平均等待时间: 2.0


常见指标：

- 等待时间。
- 服务时间。
- 系统逗留时间。
- 队列长度。
- 资源利用率。
- 完成任务数量。
- 超时或失败数量。

## 入门建模模板

实际项目可以从这个模板开始：

In [18]:
import random
import simpy


def request_process(env, name, resource, metrics):
    arrive = env.now

    with resource.request() as req:
        yield req
        start = env.now

        service_time = random.uniform(1, 3)
        print(f"{name} 开始服务")
        yield env.timeout(service_time)

        finish = env.now
        metrics.append({
            "name": name,
            "arrive": arrive,
            "start": start,
            "finish": finish,
            "wait": start - arrive,
            "service": service_time,
            "system": finish - arrive,
        })


def arrival_process(env, resource, metrics, count):
    for i in range(count):
        env.process(request_process(env, f"req-{i}", resource, metrics))
        yield env.timeout(random.expovariate(0.8))


random.seed(42)
env = simpy.Environment()
resource = simpy.Resource(env, capacity=2)
metrics = []

env.process(arrival_process(env, resource, metrics, count=10))
env.run()

print("完成数量:", len(metrics))
for row in metrics:
    print(row)
print("平均等待:", sum(row["wait"] for row in metrics) / len(metrics))

req-0 开始服务
req-1 开始服务
req-2 开始服务
req-3 开始服务
req-4 开始服务
req-5 开始服务
req-6 开始服务
req-7 开始服务
req-8 开始服务
req-9 开始服务
完成数量: 10
{'name': 'req-0', 'arrive': 0, 'start': 0, 'finish': 1.0500215104453339, 'wait': 0, 'service': 1.0500215104453339, 'system': 1.0500215104453339}
{'name': 'req-1', 'arrive': 1.2750753590935011, 'start': 1.2750753590935011, 'finish': 2.7214968353911466, 'wait': 0.0, 'service': 1.4464214762976455, 'system': 1.4464214762976455}
{'name': 'req-2', 'arrive': 1.677105439187208, 'start': 1.677105439187208, 'finish': 4.030504414033031, 'wait': 0.0, 'service': 2.3533989748458226, 'system': 2.3533989748458226}
{'name': 'req-3', 'arrive': 3.3440962801973115, 'start': 3.3440962801973115, 'finish': 4.517973945456144, 'wait': 0.0, 'service': 1.1738776652588323, 'system': 1.1738776652588325}
{'name': 'req-4', 'arrive': 6.128206404636537, 'start': 6.128206404636537, 'finish': 7.187800843512678, 'wait': 0.0, 'service': 1.0595944388761407, 'system': 1.0595944388761405}
{'name': 'req-5', '

## 入门检查清单

写 SimPy 模型时，建议逐项检查：

- 所有进程函数内部是否至少有一个 `yield`。
- 是否通过 `env.process()` 注册进程，而不是只调用生成器函数。
- 资源请求是否 `yield req`。
- 使用 `with resource.request()` 时，服务逻辑是否写在 `with` 块内。
- `env.timeout()` 的时间是否可能为负数。
- `env.run(until=...)` 的停止条件是否符合预期。
- 统计指标是否在正确的时刻记录。
- 随机数是否设置了种子，便于复现实验。